In [ ]:
import setigen as stg
import inspect

print("Setigen version:", stg.__version__)

print("\nTop-level objects:\n")

for name in sorted(dir(stg)):
    if not name.startswith("_"):
        print(name)

print("stg.Frame")
print("\nConstructor:\n")
print(inspect.signature(stg.Frame))

from blimpy import Waterfall

wf = Waterfall("/datag/pipeline/AGBT19B_999_121/blc73_blp03/blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002.h5")
frame = stg.Frame(waterfall=wf)

In [ ]:
from blimpy import Waterfall
import setigen as stg

filename = "/datag/pipeline/AGBT19B_999_121/blc73_blp03/blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002.h5"

wf = Waterfall(filename, load_data=True)

frame = stg.Frame(waterfall=wf)

print(frame)

In [ ]:
from blimpy import Waterfall
import setigen as stg

filename = "/datag/pipeline/AGBT19B_999_121/blc73_blp03/blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002.h5"

wf = Waterfall(filename, load_data=True)
frame = stg.Frame(waterfall=wf)

print("=== Frame attributes ===")

for name in sorted(dir(frame)):
    if not name.startswith("_"):
        print(name)

In [ ]:
import inspect
import setigen as stg

print(inspect.signature(stg.Frame.add_signal))
print(inspect.signature(stg.Frame.add_constant_signal))

# sweep it!

In [ ]:
from pathlib import Path
import csv
import gc
import shutil

import h5py
import numpy as np
from blimpy import Waterfall
import setigen as stg


# ============================================================
# Directories
# ============================================================

BASE = Path("/datax/scratch/wlll2x/carmen_midres")

# New directory name prevents overwriting the original
# three-cadence broad SNR sweep.
OUT_ROOT = BASE / "seticore_snr_cutoff_sweep"


# ============================================================
# Sweep settings
# ============================================================

# Focused sweep around the apparent Seticore cutoff.
SNR_VALUES = [
    20,
    25,
    30,
    35,
    40,
    45,
    50,
    60,
    75,
]

DRIFT_RATE = 0.0           # Hz/s
WIDTH = 2861               # Hz
TARGET_FREQ_HZ = 1600.0e6  # requested frequency in Hz

WINDOW = 100               # channels around injection
GUARD = 5                  # channels excluded around injection

NFPC = 1024                # channels per coarse channel

# Process cadence_000 through cadence_019.
MAX_CADENCES = 20

# Delete this cutoff sweep before regenerating it.
CLEAN_PREVIOUS_SWEEP = True


# ============================================================
# Helper functions
# ============================================================

def snr_directory_name(snr):
    """Return a sortable directory name for an SNR value."""
    if float(snr).is_integer():
        return f"snr_{int(snr):03d}"

    safe_value = str(snr).replace(".", "p")
    return f"snr_{safe_value}"


def get_local_noise_sigma(frame, channel, window, guard):
    """
    Estimate local noise using the temporal MAD of nearby
    frequency channels.
    """
    nchan = frame.data.shape[1]

    left = max(0, channel - window)
    right = min(nchan, channel + window + 1)

    local = frame.data[:, left:right]
    center = channel - left

    # Exclude the injection neighborhood.
    mask = np.ones(local.shape[1], dtype=bool)

    guard_left = max(0, center - guard)
    guard_right = min(
        local.shape[1],
        center + guard + 1,
    )

    mask[guard_left:guard_right] = False
    local = local[:, mask]

    channel_sigmas = []

    for c in range(local.shape[1]):
        values = local[:, c]

        median = np.median(values)
        mad = np.median(
            np.abs(values - median)
        )

        sigma = 1.4826 * mad

        if np.isfinite(sigma) and sigma > 0:
            channel_sigmas.append(sigma)

    channel_sigmas = np.asarray(
        channel_sigmas,
        dtype=float,
    )

    if len(channel_sigmas) == 0:
        raise RuntimeError(
            "Could not estimate local noise around the injection."
        )

    # Reject unusually quiet and unusually noisy channels.
    low = np.percentile(channel_sigmas, 10)
    high = np.percentile(channel_sigmas, 90)

    good_sigmas = channel_sigmas[
        (channel_sigmas >= low)
        & (channel_sigmas <= high)
    ]

    if len(good_sigmas) == 0:
        raise RuntimeError(
            "No local noise estimates remained after filtering."
        )

    sigma_local = float(
        np.median(good_sigmas)
    )

    return (
        sigma_local,
        len(channel_sigmas),
        left,
        right,
    )


def copy_off_with_nfpc(source, output):
    """
    Copy an unchanged OFF observation and add the nfpc metadata
    required by Seticore.
    """
    output.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if output.exists() or output.is_symlink():
        output.unlink()

    shutil.copy2(source, output)

    with h5py.File(output, "r+") as h5:
        if "data" not in h5:
            raise RuntimeError(
                f"No 'data' dataset found in OFF file: {output}"
            )

        h5["data"].attrs["nfpc"] = NFPC


def verify_saved_file(filename):
    """
    Confirm that a saved H5 contains the metadata Seticore
    requires.
    """
    with h5py.File(filename, "r") as h5:
        if "data" not in h5:
            raise RuntimeError(
                f"No 'data' dataset found in saved file: {filename}"
            )

        data = h5["data"]

        saved_nfpc = data.attrs.get("nfpc")
        saved_nchans = data.attrs.get("nchans")

        if saved_nfpc != NFPC:
            raise RuntimeError(
                f"Incorrect nfpc in {filename}: "
                f"expected {NFPC}, found {saved_nfpc}"
            )

        return (
            data.shape,
            saved_nchans,
            saved_nfpc,
        )


# ============================================================
# Prepare output directory
# ============================================================

if CLEAN_PREVIOUS_SWEEP and OUT_ROOT.exists():
    print(
        f"Removing old cutoff sweep directory: {OUT_ROOT}"
    )
    shutil.rmtree(OUT_ROOT)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

for target_snr in SNR_VALUES:
    snr_dir = (
        OUT_ROOT
        / snr_directory_name(target_snr)
    )

    snr_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# Find the first 20 cadence directories
# ============================================================

cadence_dirs = sorted(
    path
    for path in BASE.glob("cadence_*")
    if path.is_dir()
)

if MAX_CADENCES is not None:
    cadence_dirs = cadence_dirs[:MAX_CADENCES]

if not cadence_dirs:
    raise RuntimeError(
        f"No cadence directories were found under {BASE}"
    )

print("=" * 70)
print("Seticore cutoff injection sweep")
print("=" * 70)
print(f"Cadences selected : {len(cadence_dirs)}")
print(f"First cadence     : {cadence_dirs[0].name}")
print(f"Last cadence      : {cadence_dirs[-1].name}")
print(f"SNR values        : {SNR_VALUES}")
print(f"Output root       : {OUT_ROOT}")
print(f"NFPC              : {NFPC}")
print()


# ============================================================
# Summary records
# ============================================================

summary_rows = []


# ============================================================
# Process cadences
# ============================================================

for cadence_number, cadence_dir in enumerate(
    cadence_dirs,
    start=1,
):
    cadence = cadence_dir.name
    lst_file = cadence_dir / "all.lst"

    print("=" * 70)
    print(
        f"Processing {cadence} "
        f"({cadence_number}/{len(cadence_dirs)})"
    )
    print("=" * 70)

    if not lst_file.exists():
        print(f"Missing list file: {lst_file}")
        continue

    files = [
        Path(line.strip())
        for line in lst_file.read_text().splitlines()
        if line.strip()
    ]

    print(f"Found {len(files)} observations")

    if len(files) != 6:
        print(
            f"WARNING: Expected 6 observations, "
            f"but found {len(files)}."
        )

    for observation_index, filename in enumerate(files):
        inject = observation_index % 2 == 0
        scan_type = "ON" if inject else "OFF"

        print()
        print(
            f"[{observation_index + 1}/{len(files)}] "
            f"{scan_type}: {filename.name}"
        )

        if not filename.exists():
            print(f"Missing input file: {filename}")

            summary_rows.append(
                {
                    "cadence": cadence,
                    "observation_index": observation_index,
                    "scan_type": scan_type,
                    "source_file": str(filename),
                    "target_snr": "",
                    "local_noise_sigma": "",
                    "level": "",
                    "injection_frequency_MHz": "",
                    "measured_peak_snr": "",
                    "changed_pixels": "",
                    "output_file": "",
                    "status": "missing input",
                }
            )
            continue

        # ----------------------------------------------------
        # OFF scan
        # ----------------------------------------------------

        if not inject:
            for target_snr in SNR_VALUES:
                cadence_output_dir = (
                    OUT_ROOT
                    / snr_directory_name(target_snr)
                    / cadence
                )

                output = (
                    cadence_output_dir
                    / f"{filename.stem}_injected.h5"
                )

                copy_off_with_nfpc(
                    filename,
                    output,
                )

                shape, nchans, nfpc = verify_saved_file(
                    output
                )

                summary_rows.append(
                    {
                        "cadence": cadence,
                        "observation_index": observation_index,
                        "scan_type": scan_type,
                        "source_file": str(filename),
                        "target_snr": target_snr,
                        "local_noise_sigma": "",
                        "level": "",
                        "injection_frequency_MHz": "",
                        "measured_peak_snr": "",
                        "changed_pixels": 0,
                        "output_file": str(output),
                        "status": "OFF copied with nfpc",
                    }
                )

            print(
                f"Copied unchanged OFF scan into "
                f"{len(SNR_VALUES)} SNR directories."
            )
            continue

        # ----------------------------------------------------
        # Load ON observation for noise estimation
        # ----------------------------------------------------

        wf_base = Waterfall(
            str(filename),
            load_data=True,
        )

        base_frame = stg.Frame(
            waterfall=wf_base,
        )

        # ----------------------------------------------------
        # Detect Setigen frequency units
        # ----------------------------------------------------

        if base_frame.fmax > 1e7:
            frame_units = "Hz"
            frequency_scale = 1e6
            target_frequency = TARGET_FREQ_HZ
        else:
            frame_units = "MHz"
            frequency_scale = 1.0
            target_frequency = TARGET_FREQ_HZ / 1e6

        print(
            f"Setigen frequency units: {frame_units}"
        )

        print(
            "Frame frequency range: "
            f"{base_frame.fmin / frequency_scale:.6f} - "
            f"{base_frame.fmax / frequency_scale:.6f} MHz"
        )

        # ----------------------------------------------------
        # Find nearest sampled injection channel
        # ----------------------------------------------------

        channel = base_frame.get_index(
            target_frequency
        )

        f_start = base_frame.get_frequency(
            channel
        )

        if frame_units == "Hz":
            injection_hz = float(f_start)
            injection_mhz = float(f_start / 1e6)
        else:
            injection_hz = float(f_start * 1e6)
            injection_mhz = float(f_start)

        print(
            f"Requested frequency : "
            f"{TARGET_FREQ_HZ / 1e6:.9f} MHz"
        )

        print(
            f"Nearest channel     : {channel}"
        )

        print(
            f"Injection frequency : "
            f"{injection_mhz:.9f} MHz"
        )

        # ----------------------------------------------------
        # Estimate local noise once for this observation
        # ----------------------------------------------------

        (
            sigma_local,
            number_of_channels,
            local_left,
            local_right,
        ) = get_local_noise_sigma(
            frame=base_frame,
            channel=channel,
            window=WINDOW,
            guard=GUARD,
        )

        print(
            f"Local channel range : "
            f"{local_left}:{local_right}"
        )

        print(
            f"Noise channels used : "
            f"{number_of_channels}"
        )

        print(
            f"Local noise sigma   : "
            f"{sigma_local:.3e}"
        )

        del base_frame
        del wf_base
        gc.collect()

        # ----------------------------------------------------
        # Inject every requested SNR
        # ----------------------------------------------------

        for target_snr in SNR_VALUES:
            snr_name = snr_directory_name(
                target_snr
            )

            cadence_output_dir = (
                OUT_ROOT
                / snr_name
                / cadence
            )

            cadence_output_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            output = (
                cadence_output_dir
                / f"{filename.stem}_injected.h5"
            )

            # Reload the original observation so injections
            # never accumulate across SNR values.
            wf_injection = Waterfall(
                str(filename),
                load_data=True,
            )

            frame = stg.Frame(
                waterfall=wf_injection,
            )

            level = float(
                target_snr * sigma_local
            )

            before = frame.data.copy()

            signal = frame.add_constant_signal(
                f_start=f_start,
                drift_rate=DRIFT_RATE,
                level=level,
                width=WIDTH,
            )

            diff = frame.data - before

            changed_pixels = int(
                np.count_nonzero(diff)
            )

            measured_signal = signal[
                signal > 0
            ]

            if len(measured_signal) > 0:
                measured_peak = float(
                    np.max(measured_signal)
                )

                measured_peak_snr = (
                    measured_peak
                    / sigma_local
                )
            else:
                measured_peak_snr = np.nan

            print(
                f"  SNR {target_snr:>5}: "
                f"level={level:.3e}, "
                f"peak sigma={measured_peak_snr:.2f}, "
                f"changed={changed_pixels}"
            )

            if changed_pixels == 0:
                print(
                    "    WARNING: No pixels changed."
                )

            # ------------------------------------------------
            # Metadata
            # ------------------------------------------------

            frame.add_metadata(
                {
                    "injection": True,
                    "requested_frequency_Hz":
                        float(TARGET_FREQ_HZ),
                    "requested_frequency_MHz":
                        float(TARGET_FREQ_HZ / 1e6),
                    "injection_frequency_Hz":
                        injection_hz,
                    "injection_frequency_MHz":
                        injection_mhz,
                    "injection_channel":
                        int(channel),
                    "drift_rate_Hz_s":
                        float(DRIFT_RATE),
                    "width_Hz":
                        float(WIDTH),
                    "target_local_pixel_snr":
                        float(target_snr),
                    "measured_peak_snr":
                        float(measured_peak_snr),
                    "local_noise_sigma":
                        float(sigma_local),
                    "level":
                        float(level),
                    "local_window_channels":
                        int(WINDOW),
                    "local_guard_channels":
                        int(GUARD),
                    "nfpc":
                        int(NFPC),
                }
            )

            # ------------------------------------------------
            # Save
            # ------------------------------------------------

            frame.save_hdf5(
                str(output)
            )

            # Guarantee nfpc exists where Seticore expects it.
            with h5py.File(output, "r+") as h5:
                h5["data"].attrs["nfpc"] = NFPC

            shape, nchans, nfpc = verify_saved_file(
                output
            )

            print(
                f"           saved shape={shape}, "
                f"nchans={nchans}, nfpc={nfpc}"
            )

            summary_rows.append(
                {
                    "cadence": cadence,
                    "observation_index": observation_index,
                    "scan_type": scan_type,
                    "source_file": str(filename),
                    "target_snr": target_snr,
                    "local_noise_sigma": sigma_local,
                    "level": level,
                    "injection_frequency_MHz": injection_mhz,
                    "measured_peak_snr": measured_peak_snr,
                    "changed_pixels": changed_pixels,
                    "output_file": str(output),
                    "status": "injected with nfpc",
                }
            )

            del measured_signal
            del signal
            del diff
            del before
            del frame
            del wf_injection

            gc.collect()


# ============================================================
# Save sweep manifest
# ============================================================

manifest = (
    OUT_ROOT
    / "injection_manifest.csv"
)

fieldnames = [
    "cadence",
    "observation_index",
    "scan_type",
    "source_file",
    "target_snr",
    "local_noise_sigma",
    "level",
    "injection_frequency_MHz",
    "measured_peak_snr",
    "changed_pixels",
    "output_file",
    "status",
]

with manifest.open("w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=fieldnames,
    )

    writer.writeheader()
    writer.writerows(summary_rows)


# ============================================================
# Final summary
# ============================================================

injected_count = sum(
    row["status"] == "injected with nfpc"
    for row in summary_rows
)

off_count = sum(
    row["status"] == "OFF copied with nfpc"
    for row in summary_rows
)

missing_count = sum(
    row["status"] == "missing input"
    for row in summary_rows
)

expected_on = (
    len(cadence_dirs)
    * 3
    * len(SNR_VALUES)
)

expected_off = (
    len(cadence_dirs)
    * 3
    * len(SNR_VALUES)
)

print()
print("=" * 70)
print("CUTOFF SWEEP COMPLETE")
print("=" * 70)
print(f"Injected ON files : {injected_count}")
print(f"Expected ON files : {expected_on}")
print(f"Copied OFF files  : {off_count}")
print(f"Expected OFF files: {expected_off}")
print(f"Missing inputs    : {missing_count}")
print(f"Manifest          : {manifest}")
print(f"Output root       : {OUT_ROOT}")
print(f"NFPC              : {NFPC}")

In [ ]:
wf = Waterfall(
    "/datag/pipeline/AGBT19B_999_121/blc73_blp03/blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002.h5",
    load_data=True
)

frame = stg.Frame(waterfall=wf)

print("Before:")
print(frame.data.max())

signal = frame.add_constant_signal(
    f_start=frame.get_frequency(frame.fchans//2),
    drift_rate=0,
    level=100,
    width=5000,
)

print("\nSignal:")
print(signal.max())
print(signal.shape)

print("\nAfter:")
print(frame.data.max())

In [ ]:
before = frame.data.copy()

signal = frame.add_constant_signal(
    f_start=frame.get_frequency(frame.fchans//2),
    drift_rate=0,
    level=100,
    width=5000,
)

print("Before add:")
print(np.count_nonzero(frame.data - before))

frame.data += signal

after = frame.data

print("After manual add:")
print(np.count_nonzero(after - before))

In [ ]:
signal = frame.add_constant_signal(
    f_start=frame.get_frequency(frame.fchans//2),
    drift_rate=0,
    level=100,
    width=5000,
)

print("signal dtype:", signal.dtype)
print("frame dtype:", frame.data.dtype)

print("signal min:", signal.min())
print("signal max:", signal.max())

print("nonzero signal pixels:", np.count_nonzero(signal))

print("frame max before:", frame.data.max())

frame.data += signal

print("frame max after:", frame.data.max())

print("difference max:", np.max(frame.data - before))
print("difference nonzero:", np.count_nonzero(frame.data - before))

print("Data statistics:")
print("min:", frame.data.min())
print("max:", frame.data.max())
print("median:", np.median(frame.data))
print("std:", np.std(frame.data))

In [ ]:
from blimpy import Waterfall
wf=Waterfall('/datax/scratch/wlll2x/carmen_midres/injected_test/blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002_injected.h5', load_data=False)
print(wf.header['fch1'])
print(wf.header['foff'])
print(wf.header['tsamp'])
print(wf.container.selection_shape)


In [ ]:
from blimpy import Waterfall

wf = Waterfall(
    "/datax/scratch/wlll2x/carmen_midres/injected_test/blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002_injected.h5",
    load_data=False
)

print("Band:")
print("fmin =", wf.container.f_start)
print("fmax =", wf.container.f_stop)

# Plot

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from blimpy import Waterfall

# ============================================================
# Files
# ============================================================

original = Path(
    "/datag/pipeline/AGBT19B_999_121/blc73_blp03/"
    "blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002.h5"
)

injected = Path(
    "/datax/scratch/wlll2x/carmen_midres/injected_test/"
    "blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002_injected.h5"
)

# ============================================================
# Load
# ============================================================

wf_orig = Waterfall(str(original), load_data=True)
wf_inj = Waterfall(str(injected), load_data=True)

orig = wf_orig.data.squeeze()
inj = wf_inj.data.squeeze()

print(f"Original shape : {orig.shape}")
print(f"Injected shape : {inj.shape}")

# ============================================================
# Difference
# ============================================================

diff = inj - orig

print(f"Maximum difference : {diff.max():.3e}")
print(f"Changed pixels     : {np.count_nonzero(diff)}")

# ============================================================
# Frequency axis (already in MHz)
# ============================================================

fch1 = float(wf_orig.header["fch1"])
foff = float(wf_orig.header["foff"])

nchan = orig.shape[1]

freqs = fch1 + np.arange(nchan, dtype=np.float64) * foff

print(f"Header fch1 : {fch1:.9f} MHz")
print(f"Header foff : {foff:.12f} MHz/channel")
print(f"Frequency range : {freqs.min():.6f} - {freqs.max():.6f} MHz")

# ============================================================
# Locate strongest injected pixel
# ============================================================

time_index, freq_index = np.unravel_index(
    np.nanargmax(diff),
    diff.shape
)

inj_freq = freqs[freq_index]

print(f"Peak channel    : {freq_index}")
print(f"Peak frequency  : {inj_freq:.9f} MHz")
print(f"Offset from 1600 MHz : {(inj_freq-1600.0)*1e6:.1f} Hz")

# ============================================================
# Zoom around injection
# ============================================================

CHANNEL_WINDOW = 100

lo = max(freq_index - CHANNEL_WINDOW, 0)
hi = min(freq_index + CHANNEL_WINDOW + 1, nchan)

orig_zoom = orig[:, lo:hi]
inj_zoom = inj[:, lo:hi]
diff_zoom = diff[:, lo:hi]

freq_zoom = freqs[lo:hi]

if freq_zoom.size == 0:
    raise RuntimeError("Zoom window is empty.")

print(f"Zoom channels : {lo} - {hi-1}")
print(
    f"Zoom frequency range : "
    f"{freq_zoom.min():.6f} - {freq_zoom.max():.6f} MHz"
)

# ============================================================
# Make frequency increase left -> right
# ============================================================

if freq_zoom[0] > freq_zoom[-1]:
    freq_zoom = freq_zoom[::-1]
    orig_zoom = orig_zoom[:, ::-1]
    inj_zoom = inj_zoom[:, ::-1]
    diff_zoom = diff_zoom[:, ::-1]

# ============================================================
# Display scaling
# ============================================================

vmin = np.percentile(orig_zoom, 5)
vmax = np.percentile(orig_zoom, 99.9)

positive = diff_zoom[diff_zoom > 0]

if positive.size > 0:
    diff_vmax = np.percentile(positive, 99)
else:
    diff_vmax = diff_zoom.max()

# ============================================================
# Time axis
# ============================================================

dt = float(wf_orig.header["tsamp"])
obs_time = orig.shape[0] * dt

print(f"Integration time : {dt:.6f} s")
print(f"Observation time : {obs_time:.3f} s")

extent = [
    freq_zoom[0],
    freq_zoom[-1],
    obs_time,
    0,
]

peak_time = time_index * dt

# ============================================================
# Plot
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6),
    sharey=True,
)

# ------------------------------------------------------------
# Original
# ------------------------------------------------------------

im0 = axes[0].imshow(
    orig_zoom,
    aspect="auto",
    origin="upper",
    extent=extent,
    vmin=vmin,
    vmax=vmax,
)

axes[0].set_title("Original")
axes[0].set_xlabel("Frequency (MHz)")
axes[0].set_ylabel("Time (s)")

plt.colorbar(im0, ax=axes[0], label="Power")

# ------------------------------------------------------------
# Injected
# ------------------------------------------------------------

im1 = axes[1].imshow(
    inj_zoom,
    aspect="auto",
    origin="upper",
    extent=extent,
    vmin=vmin,
    vmax=vmax,
)

axes[1].set_title("Injected")
axes[1].set_xlabel("Frequency (MHz)")

plt.colorbar(im1, ax=axes[1], label="Power")

# ------------------------------------------------------------
# Difference
# ------------------------------------------------------------

im2 = axes[2].imshow(
    diff_zoom,
    aspect="auto",
    origin="upper",
    extent=extent,
    vmin=0,
    vmax=diff_vmax,
)

axes[2].set_title("Injected − Original")
axes[2].set_xlabel("Frequency (MHz)")

plt.colorbar(im2, ax=axes[2], label="Injected Power")

# ============================================================
# Mark detected injection
# ============================================================

for ax in axes:
    ax.axvline(
        inj_freq,
        color="white",
        linestyle="--",
        linewidth=1,
    )

    ax.plot(
        inj_freq,
        peak_time,
        marker="+",
        color="red",
        markersize=10,
        markeredgewidth=2,
    )

plt.tight_layout()
plt.show()

In [ ]:
nchan = orig.shape[1]

freqs = fch1 + np.arange(nchan, dtype=np.float64) * foff

print("First frequency :", freqs[0])
print("Last frequency  :", freqs[-1])
print("Min frequency   :", freqs.min())
print("Max frequency   :", freqs.max())

time_index, freq_index = np.unravel_index(np.argmax(diff), diff.shape)

print("Peak channel:", freq_index)
print("Peak frequency:", freqs[freq_index])

In [ ]:
from blimpy import Waterfall

wf = Waterfall(
    "/datax/scratch/wlll2x/carmen_midres/injected_test/"
    "blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002_injected.h5",
    load_data=True
)

print("fch1 =", wf.header["fch1"])
print("foff =", wf.header["foff"])
print("data shape =", wf.data.shape)

In [ ]:
from blimpy import Waterfall
import numpy as np

orig = Waterfall(
    "/datag/pipeline/AGBT19B_999_121/blc73_blp03/blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002.h5"
)

inj = Waterfall(
    "/datax/scratch/wlll2x/carmen_midres/injected_test/blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002_injected.h5"
)

diff = inj.data - orig.data

print("Maximum difference:", np.max(diff))
print("Minimum difference:", np.min(diff))
print("Number of changed pixels:", np.count_nonzero(diff))

# Plot injected cadence

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from blimpy import Waterfall


# ==========================================================
# Paths
# ==========================================================

H5_DIR = Path(
    "/datax/scratch/wlll2x/carmen_midres/injected_lower"
)

DAT_DIR = Path(
    "/datax/scratch/wlll2x/carmen_midres_bliss_injected_lower_combined"
)


# ==========================================================
# Parameters
# ==========================================================

target_freq = 1600.0   # MHz
zoom_MHz = 0.05
FILES_PER_CADENCE = 6


# ==========================================================
# BLISS parser
# ==========================================================

def read_bliss_dat(filename):

    tables = []

    with open(filename, "r") as f:

        header = None
        rows = []

        for line in f:

            clean = line.strip()

            if clean.startswith("#"):
                clean = clean[1:].strip()


            if "Top_Hit" in clean and "SNR" in clean:

                if header and rows:
                    tables.append(
                        pd.DataFrame(
                            rows,
                            columns=header
                        )
                    )

                header = clean.split()
                rows = []

                continue


            if clean.startswith("-"):
                continue


            if header and clean and clean[0].isdigit():

                vals = clean.split()[:len(header)]

                if len(vals) == len(header):

                    rows.append(vals)


    if header and rows:

        tables.append(
            pd.DataFrame(
                rows,
                columns=header
            )
        )


    if not tables:
        return None


    hits = pd.concat(
        tables,
        ignore_index=True
    )


    for c in hits.columns:

        hits[c] = pd.to_numeric(
            hits[c],
            errors="coerce"
        )


    return hits



# ==========================================================
# Files
# ==========================================================

h5_files = sorted(
    H5_DIR.glob("*_injected.h5")
)

dat_files = sorted(
    DAT_DIR.glob("*_combined.dat")
)


print("Total H5 :", len(h5_files))
print("Total DAT:", len(dat_files))


# ==========================================================
# Plot cadences
# ==========================================================

for cadence in range(3):


    print("\n" + "="*60)
    print(
        f"CADENCE {cadence:03d}"
    )
    print("="*60)


    start = cadence * FILES_PER_CADENCE
    end = start + FILES_PER_CADENCE


    cadence_h5 = h5_files[start:end]
    cadence_dat = dat_files[start:end]


    print("\nH5 files:")
    for f in cadence_h5:
        print(f.name)


    print("\nDAT files:")
    for f in cadence_dat:
        print(f.name)


    if len(cadence_h5) == 0:

        print("No H5 files found")
        continue


    if len(cadence_dat) == 0:

        print("No DAT files found")
        continue



    # ------------------------------------------------------
    # OFF-only scaling
    # ------------------------------------------------------

    off_pixels = []


    for j, h5 in enumerate(cadence_h5):

        if j % 2 == 1:


            wf = Waterfall(
                str(h5),
                load_data=True
            )


            data = np.squeeze(
                wf.data
            )


            freqs = (
                float(wf.header["fch1"])
                +
                np.arange(data.shape[1])
                *
                float(wf.header["foff"])
            )


            mask = (
                (freqs > target_freq-zoom_MHz)
                &
                (freqs < target_freq+zoom_MHz)
            )


            crop = data[:,mask]


            image = 10*np.log10(
                crop - crop.min() + 1
            )


            off_pixels.extend(
                image.flatten()
            )


    if len(off_pixels) == 0:

        print(
            "No OFF pixels found, skipping"
        )

        continue


    off_pixels = np.array(
        off_pixels
    )


    vmin = np.percentile(
        off_pixels,
        5
    )

    vmax = np.percentile(
        off_pixels,
        99
    )


    print(
        "Scale:",
        vmin,
        vmax
    )


    fig, axes = plt.subplots(
        6,
        1,
        figsize=(14,18),
        sharex=True
    )


    # IMPORTANT FIX
    im = None



    # ------------------------------------------------------
    # Individual scans
    # ------------------------------------------------------

    for i, (ax, h5, dat) in enumerate(
        zip(
            axes,
            cadence_h5,
            cadence_dat
        )
    ):


        wf = Waterfall(
            str(h5),
            load_data=True
        )


        data = np.squeeze(
            wf.data
        )


        freqs = (
            float(wf.header["fch1"])
            +
            np.arange(data.shape[1])
            *
            float(wf.header["foff"])
        )


        inj_idx = np.argmin(
            abs(freqs-target_freq)
        )


        inj_freq = freqs[inj_idx]


        mask = (
            (freqs > target_freq-zoom_MHz)
            &
            (freqs < target_freq+zoom_MHz)
        )


        crop = data[:,mask]

        crop_freqs = freqs[mask]


        image = 10*np.log10(
            crop - crop.min() + 1
        )


        times = (
            np.arange(data.shape[0])
            *
            float(wf.header["tsamp"])
        )



        # --------------------------------------------------
        # Waterfall
        # --------------------------------------------------

        im = ax.imshow(
            image,
            aspect="auto",
            origin="upper",
            interpolation="none",
            extent=[
                crop_freqs.max(),
                crop_freqs.min(),
                times[-1],
                times[0]
            ],
            vmin=vmin,
            vmax=vmax
        )


        ax.axvline(
            inj_freq,
            linestyle="--",
            linewidth=1
        )



        # --------------------------------------------------
        # BLISS
        # --------------------------------------------------

        hits = read_bliss_dat(dat)


        if hits is not None:


            nearby = hits[
                abs(
                    hits["Corrected_Frequency"]
                    -
                    target_freq
                )
                <
                zoom_MHz
            ].sort_values(
                "SNR",
                ascending=False
            )


            if len(nearby):


                print("\n", h5.name)

                print(
                    nearby[
                        [
                            "Corrected_Frequency",
                            "Drift_Rate",
                            "SNR"
                        ]
                    ]
                )



                for _, row in nearby.iterrows():


                    freq0 = row["Corrected_Frequency"]


                    drift = (
                        row["Drift_Rate"]
                        /
                        1e6
                    )


                    drift_track = (
                        freq0
                        +
                        drift *
                        (
                            times
                            -
                            times[len(times)//2]
                        )
                    )


                    ax.plot(
                        drift_track,
                        times,
                        ":",
                        linewidth=2.5,
                        label=
                        f"{row['Drift_Rate']:.2f} Hz/s"
                    )


                    ax.scatter(
                        freq0,
                        times[len(times)//2],
                        marker="x",
                        s=100
                    )


                ax.legend(
                    fontsize=10
                )



        # --------------------------------------------------
        # Labels
        # --------------------------------------------------

        ax.text(
            0.02,
            0.82,
            "ON" if i % 2 == 0 else "OFF",
            transform=ax.transAxes,
            fontsize=20,
            fontweight="bold",
            bbox=dict(
                facecolor="white",
                alpha=0.8
            )
        )


        ax.set_ylabel(
            "Time (s)"
        )



    axes[-1].set_xlabel(
        "Frequency (MHz)",
        fontsize=14
    )


    # IMPORTANT FIX
    if im is not None:

        fig.colorbar(
            im,
            ax=axes,
            label="Power (dB)"
        )

    else:

        print(
            "No image generated for colorbar"
        )



    plt.suptitle(
        f"BLISS Zero Drift Injection Lower\nCadence {cadence:03d}",
        fontsize=20,
        fontweight="bold"
    )


    plt.tight_layout()

    plt.show()

In [ ]:
from pathlib import Path
import pandas as pd


# ==========================================================
# Pick a BLISS .dat file
# ==========================================================

DAT_DIR = Path(
    "/datax/scratch/wlll2x/carmen_midres_bliss_injected_combined"
)


dat_files = sorted(
    DAT_DIR.glob("*.dat")
)


print(f"Found {len(dat_files)} dat files\n")


for i, f in enumerate(dat_files[:10]):
    print(i, f.name)


# ==========================================================
# Select first file
# ==========================================================

dat_file = dat_files[0]

print("\n======================================")
print("Testing:")
print(dat_file)
print("======================================\n")


# ==========================================================
# Print raw contents
# ==========================================================

print("FIRST 30 LINES")
print("--------------------------------------")


with open(dat_file, "r") as f:

    for i, line in enumerate(f):

        print(
            f"{i:03d}: {repr(line)}"
        )

        if i >= 29:
            break


# ==========================================================
# Try different pandas strategies
# ==========================================================

print("\n\n======================================")
print("Attempt 1: whitespace parser")
print("======================================")

try:

    df = pd.read_csv(
        dat_file,
        sep=r"\s+",
        engine="python"
    )

    print("SUCCESS")

    print(
        df.head()
    )

    print(
        df.columns
    )


except Exception as e:

    print("FAILED")
    print(e)



print("\n\n======================================")
print("Attempt 2: ignore # comments")
print("======================================")


try:

    df = pd.read_csv(
        dat_file,
        sep=r"\s+",
        engine="python",
        comment="#"
    )


    print("SUCCESS")

    print(
        df.head()
    )

    print(
        df.columns
    )


except Exception as e:

    print("FAILED")
    print(e)



print("\n\n======================================")
print("Attempt 3: python engine + skip bad lines")
print("======================================")


try:

    df = pd.read_csv(
        dat_file,
        sep=r"\s+",
        engine="python",
        on_bad_lines="skip"
    )


    print("SUCCESS")

    print(
        df.head()
    )

    print(
        df.columns
    )


except Exception as e:

    print("FAILED")
    print(e)



In [ ]:
from pathlib import Path

dat_file = Path(
    "/datax/scratch/wlll2x/carmen_midres_bliss_injected_combined/"
    "blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002_injected_combined.dat"
)

with open(dat_file, "r") as f:
    for i in range(40):
        print(f"{i:03d}: {f.readline().rstrip()}")

In [ ]:
def read_bliss_dat(filename):

    """
    Reads concatenated BLISS .dat files.
    Handles extra columns in BLISS output.
    """

    tables = []

    with open(filename, "r") as f:
        lines = f.readlines()


    header = None
    rows = []


    for line in lines:

        clean = line.strip()


        if clean.startswith("#"):
            clean = clean[1:].strip()


        # Find header

        if (
            "Top_Hit" in clean
            and
            "SNR" in clean
        ):

            if header is not None and rows:

                tables.append(
                    pd.DataFrame(
                        rows,
                        columns=header
                    )
                )


            header = clean.split()

            rows = []

            continue



        if clean.startswith("-"):
            continue



        if (
            header is not None
            and clean
            and clean[0].isdigit()
        ):

            values = clean.split()


            # BLISS sometimes has extra columns
            values = values[:len(header)]


            # only accept valid rows

            if len(values) == len(header):

                rows.append(values)



    # final block

    if header is not None and rows:

        tables.append(
            pd.DataFrame(
                rows,
                columns=header
            )
        )



    if not tables:
        return None



    hits = pd.concat(
        tables,
        ignore_index=True
    )


    # convert numbers

    for col in hits.columns:

        hits[col] = pd.to_numeric(
            hits[col],
            errors="coerce"
        )


    return hits

# extract coarse

In [ ]:
from blimpy import Waterfall
import numpy as np

filename = "/datax/scratch/wlll2x/carmen_midres/injected_test/blc73_guppi_58832_08450_NGC7640_0033.gpuspec.0002_injected.h5"

wf = Waterfall(filename, load_data=False)

fch1 = float(wf.header["fch1"])
foff = float(wf.header["foff"])

target_freq = 1600.0  # MHz

channel = int(round((target_freq - fch1) / foff))
coarse = channel // 1024

print("Target channel:", channel)
print("Coarse channel:", coarse)
print("Channel within coarse:", channel % 1024)

In [ ]:
import h5py
import os
import glob

# ==========================================================
# Input / Output
# ==========================================================

indir = "/datax/scratch/wlll2x/carmen_midres/injected_test"
outdir = "/datax/scratch/wlll2x/seticore_test"

os.makedirs(outdir, exist_ok=True)

# ==========================================================
# Injection coarse channel
# ==========================================================

COARSE = 30
NFPC = 1024

start = COARSE * NFPC
stop = (COARSE + 1) * NFPC

print(f"Extracting coarse channel {COARSE}")
print(f"Channels {start} : {stop-1}")

# ==========================================================
# Extract each injected file
# ==========================================================

for filename in sorted(glob.glob(os.path.join(indir, "*.h5"))):

    print("\n=======================================")
    print("Processing:", os.path.basename(filename))
    print("=======================================")

    outfile = os.path.join(
        outdir,
        os.path.basename(filename).replace(
            ".h5",
            f"_coarse{COARSE}.h5"
        )
    )

    with h5py.File(filename, "r") as fin:
        with h5py.File(outfile, "w") as fout:

            # --------------------------------------------------
            # Copy root attributes
            # --------------------------------------------------

            for key, val in fin.attrs.items():
                fout.attrs[key] = val

            # --------------------------------------------------
            # Crop data
            # --------------------------------------------------

            data = fin["data"][:, :, start:stop]

            d = fout.create_dataset(
                "data",
                data=data,
                dtype=fin["data"].dtype
            )

            # Copy all existing attributes
            for key, val in fin["data"].attrs.items():
                d.attrs[key] = val

            # --------------------------------------------------
            # Update metadata
            # --------------------------------------------------

            d.attrs["nchans"] = NFPC
            d.attrs["nfpc"] = NFPC

            d.attrs["fch1"] = (
                fin["data"].attrs["fch1"]
                +
                start * fin["data"].attrs["foff"]
            )

            print("New fch1:", d.attrs["fch1"])
            print("nchans :", d.attrs["nchans"])
            print("nfpc   :", d.attrs["nfpc"])

            # --------------------------------------------------
            # Copy mask if present
            # --------------------------------------------------

            if "mask" in fin:

                mask = fin["mask"][:, :, start:stop]

                m = fout.create_dataset(
                    "mask",
                    data=mask
                )

                for key, val in fin["mask"].attrs.items():
                    m.attrs[key] = val

    print("Saved:", outfile)

print("\n=======================================")
print("Finished extracting coarse channel 30")
print("=======================================")

# Use find_event

In [ ]:
from pathlib import Path
from turbo_seti.find_event.find_event_pipeline import find_event_pipeline

# ==========================================================
# Directories
# ==========================================================

base_dir = Path("/datax/scratch/wlll2x/seticore_test")

results_dir = base_dir / "results"
h5_dir = base_dir

dat_list = base_dir / "seticore_test_dat.lst"
h5_list = base_dir / "seticore_test_h5.lst"

# ==========================================================
# Gather files
# ==========================================================

dat_files = sorted(results_dir.glob("*.dat"))

h5_files = sorted(
    f for f in h5_dir.glob("*.h5")
    if not f.name.startswith("voyager")
)

print(f"Found {len(dat_files)} dat files")
print(f"Found {len(h5_files)} h5 files")

assert len(dat_files) == 18, f"Expected 18 .dat files, found {len(dat_files)}"
assert len(h5_files) == 18, f"Expected 18 .h5 files, found {len(h5_files)}"

print("\nPairing:")
for dat, h5 in zip(dat_files, h5_files):
    print(f"{dat.name:<45} <---> {h5.name}")
# ==========================================================
# Write list files
# ==========================================================

with open(dat_list, "w") as f:
    for file in dat_files:
        f.write(str(file.resolve()) + "\n")

with open(h5_list, "w") as f:
    for file in h5_files:
        f.write(str(file.resolve()) + "\n")

print("List files written.")

# ==========================================================
# Run find_event
# ==========================================================

events = find_event_pipeline(
    dat_file_list_str=str(dat_list),
    h5_file_list_str=str(h5_list),

    number_in_cadence=6,
    on_off_first="ON",

    check_zero_drift=True,
    filter_threshold=1,

    saving=True,
    csv_name=str(base_dir / "seticore_events.csv")
)

print(events)

In [ ]:
from pathlib import Path

base_dir = Path("/datax/scratch/wlll2x/seticore_test")

h5_files = sorted(base_dir.glob("*.h5"))

print(len(h5_files))

for f in h5_files:
    print(f.name)

In [ ]:
from pathlib import Path
from turbo_seti.find_event.find_event_pipeline import find_event_pipeline

# directories

base_dir = Path("/datax/scratch/wlll2x/seticore_lower")

results_dir = base_dir / "results"
h5_dir = base_dir

dat_list = base_dir / "seticore_test_dat.lst"
h5_list = base_dir / "seticore_test_h5.lst"

# ==========================================================
# Gather files
# ==========================================================

dat_files = sorted(results_dir.glob("*.dat"))

h5_files = sorted(
    f for f in h5_dir.glob("*.h5")
    if not f.name.startswith("voyager")
)

print(f"Found {len(dat_files)} dat files")
print(f"Found {len(h5_files)} h5 files")

assert len(dat_files) == 18, f"Expected 18 .dat files, found {len(dat_files)}"
assert len(h5_files) == 18, f"Expected 18 .h5 files, found {len(h5_files)}"

print("\nPairing:")
for dat, h5 in zip(dat_files, h5_files):
    print(f"{dat.name:<45} <---> {h5.name}")
# ==========================================================
# Write list files
# ==========================================================

with open(dat_list, "w") as f:
    for file in dat_files:
        f.write(str(file.resolve()) + "\n")

with open(h5_list, "w") as f:
    for file in h5_files:
        f.write(str(file.resolve()) + "\n")

print("List files written.")

# run!

events = find_event_pipeline(
    dat_file_list_str=str(dat_list),
    h5_file_list_str=str(h5_list),

    number_in_cadence=6,
    on_off_first="ON",

    check_zero_drift=True,
    filter_threshold=1,

    saving=True,
    csv_name=str(base_dir / "seticore_events.csv")
)

print(events)

# in_under

In [ ]:
import pandas as pd

events = pd.read_csv(
    "/datax/scratch/wlll2x/seticore_test/seticore_events.csv"
)

print(events.columns)

In [ ]:
events[
    [
        "Freq",
        "DriftRate",
        "SNR",
        "status",
        "FileID"
    ]
].sort_values(
    "SNR",
    ascending=False
)

# heckin plot the sweep!!

In [ ]:
%matplotlib inline

from pathlib import Path
import shutil
import warnings

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from blimpy import Waterfall
from turbo_seti.find_event.find_event_pipeline import find_event_pipeline


# ============================================================
# Main directories
# ============================================================

BASE = Path("/datax/scratch/wlll2x/carmen_midres")

H5_ROOT = BASE / "seticore_snr_sweep"
DAT_ROOT = BASE / "seticore_snr_results"

EVENT_ROOT = BASE / "seticore_snr_events"
FIGURE_ROOT = BASE / "seticore_snr_figures"


# ============================================================
# Output cleanup
# ============================================================

CLEAR_EVENT_ROOT = True
CLEAR_FIGURE_ROOT = True

if CLEAR_EVENT_ROOT and EVENT_ROOT.exists():
    print(f"Removing old event directory: {EVENT_ROOT}")
    shutil.rmtree(EVENT_ROOT)

if CLEAR_FIGURE_ROOT and FIGURE_ROOT.exists():
    print(f"Removing old figure directory: {FIGURE_ROOT}")
    shutil.rmtree(FIGURE_ROOT)

EVENT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# General parameters
# ============================================================

FILES_PER_CADENCE = 6

TARGET_FREQ_MHZ = 1600.0
ZOOM_MHZ = 0.05

# Maximum distance allowed between a find_event candidate
# and the injected frequency.
EVENT_MATCH_TOLERANCE_MHZ = 0.01

# Fixed display limits in robust OFF-noise sigma units.
VMIN_SIGMA = -5
VMAX_SIGMA = 15

# Run find_event again even if an old CSV exists.
RERUN_FIND_EVENT = True

# Save PNG figures in addition to displaying them.
SAVE_FIGURES = True

# Use a short list during testing if desired:
# PLOT_SNRS = [5, 30, 50, 500]

PLOT_SNRS = [
    5,
    10,
    15,
    20,
    30,
    40,
    50,
    75,
    100,
    150,
    200,
    300,
    400,
    500,
]


# ============================================================
# Plot styling
# ============================================================

FIGURE_WIDTH = 17
FIGURE_HEIGHT = 22

TITLE_SIZE = 25
EVENT_LABEL_SIZE = 14

AXIS_LABEL_SIZE = 19
TICK_LABEL_SIZE = 15

SCAN_LABEL_SIZE = 21
SOURCE_LABEL_SIZE = 15

LEGEND_SIZE = 13
COLORBAR_LABEL_SIZE = 18

INJECTION_LINEWIDTH = 2.2
EVENT_LINEWIDTH = 3.0
EVENT_MARKER_SIZE = 170


# ============================================================
# Helper functions
# ============================================================

def snr_directory_name(snr):
    """Convert an SNR value into the sweep directory format."""
    if float(snr).is_integer():
        return f"snr_{int(snr):03d}"

    safe_value = str(snr).replace(".", "p")
    return f"snr_{safe_value}"


def expected_stem(original_file):
    """
    Return the stem shared by the injected H5 file and
    corresponding Seticore DAT file.
    """
    return f"{original_file.stem}_injected"


def get_ordered_cadence_files(snr_name, cadence_name):
    """
    Use the original cadence all.lst file to preserve the exact
    scan order:

        ON, OFF, ON, OFF, ON, OFF
    """
    original_cadence_dir = BASE / cadence_name
    lst_file = original_cadence_dir / "all.lst"

    if not lst_file.exists():
        raise FileNotFoundError(
            f"Missing original cadence list: {lst_file}"
        )

    original_files = [
        Path(line.strip())
        for line in lst_file.read_text().splitlines()
        if line.strip()
    ]

    if len(original_files) != FILES_PER_CADENCE:
        raise RuntimeError(
            f"{cadence_name}: expected {FILES_PER_CADENCE} "
            f"files in all.lst, found {len(original_files)}"
        )

    h5_dir = H5_ROOT / snr_name / cadence_name
    dat_dir = DAT_ROOT / snr_name / cadence_name

    h5_files = []
    dat_files = []

    for original_file in original_files:
        stem = expected_stem(original_file)

        h5_files.append(
            h5_dir / f"{stem}.h5"
        )

        dat_files.append(
            dat_dir / f"{stem}.dat"
        )

    return h5_files, dat_files


def validate_file_pairing(h5_files, dat_files):
    """Check that all expected H5 and DAT files exist and match."""
    problems = []

    for index, (h5_file, dat_file) in enumerate(
        zip(h5_files, dat_files)
    ):
        scan_type = "ON" if index % 2 == 0 else "OFF"

        if not h5_file.exists():
            problems.append(
                f"Missing H5: {h5_file}"
            )

        if not dat_file.exists():
            problems.append(
                f"Missing DAT: {dat_file}"
            )

        if h5_file.stem != dat_file.stem:
            problems.append(
                f"Stem mismatch: "
                f"{h5_file.name} versus {dat_file.name}"
            )

        print(
            f"{index + 1}: {scan_type:<3}  "
            f"{h5_file.name:<65} <--> {dat_file.name}"
        )

    if problems:
        raise FileNotFoundError(
            "\n".join(problems)
        )


def write_file_list(files, output_file):
    """Write absolute paths to a list file for find_event."""
    with output_file.open("w") as handle:
        for filename in files:
            handle.write(
                str(filename.resolve()) + "\n"
            )


def find_column(dataframe, possible_names):
    """
    Find a dataframe column while tolerating capitalization
    and small naming differences.
    """
    normalized = {
        str(column).strip().lower(): column
        for column in dataframe.columns
    }

    for possible_name in possible_names:
        key = possible_name.lower()

        if key in normalized:
            return normalized[key]

    return None


def load_events(csv_file):
    """Load a find_event CSV without failing on empty files."""
    if not csv_file.exists():
        return pd.DataFrame()

    try:
        return pd.read_csv(csv_file)

    except pd.errors.EmptyDataError:
        return pd.DataFrame()

    except Exception as error:
        warnings.warn(
            f"Could not read {csv_file}: {error}"
        )
        return pd.DataFrame()


def get_injection_frequency(h5_files):
    """
    Read the injected frequency from the first ON file.

    Fall back to TARGET_FREQ_MHZ if the injection metadata
    cannot be found.
    """
    for index, filename in enumerate(h5_files):
        if index % 2 != 0:
            continue

        try:
            with h5py.File(filename, "r") as h5:
                attrs = h5["data"].attrs

                if "injection_frequency_MHz" in attrs:
                    return float(
                        attrs["injection_frequency_MHz"]
                    )

                if "injection_frequency_Hz" in attrs:
                    return (
                        float(
                            attrs["injection_frequency_Hz"]
                        )
                        / 1e6
                    )

        except Exception as error:
            warnings.warn(
                f"Could not read injection metadata from "
                f"{filename}: {error}"
            )

    return TARGET_FREQ_MHZ


def load_waterfall_crop(filename, target_freq, zoom_mhz):
    """
    Load one H5 file and return the raw cropped waterfall.

    No logarithmic conversion or independent panel scaling is
    performed here.
    """
    wf = Waterfall(
        str(filename),
        load_data=True,
    )

    data = np.squeeze(
        wf.data
    ).astype(np.float64)

    if data.ndim != 2:
        raise RuntimeError(
            f"Expected two-dimensional data, found "
            f"{data.shape} in {filename}"
        )

    fch1 = float(wf.header["fch1"])
    foff = float(wf.header["foff"])
    tsamp = float(wf.header["tsamp"])

    source_name = str(
        wf.header.get(
            "source_name",
            filename.stem,
        )
    )

    frequencies = (
        fch1
        + np.arange(data.shape[1]) * foff
    )

    frequency_mask = (
        (frequencies >= target_freq - zoom_mhz)
        & (frequencies <= target_freq + zoom_mhz)
    )

    if not np.any(frequency_mask):
        raise RuntimeError(
            f"No channels found around "
            f"{target_freq:.6f} MHz in {filename.name}"
        )

    crop = data[:, frequency_mask]
    crop_frequencies = frequencies[frequency_mask]

    times = (
        np.arange(data.shape[0])
        * tsamp
    )

    return (
        crop,
        crop_frequencies,
        times,
        source_name,
    )


def remove_broadband_background(crop):
    """
    Remove broadband time-dependent structure.

    At each time integration, subtract the median across the
    plotted frequency window. A narrowband signal remains
    because it occupies only a small number of channels.
    """
    time_median = np.nanmedian(
        crop,
        axis=1,
        keepdims=True,
    )

    return crop - time_median


def robust_sigma(values):
    """
    Estimate Gaussian-equivalent sigma using the median
    absolute deviation.
    """
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        raise RuntimeError(
            "Cannot estimate noise from an empty array."
        )

    median = np.median(values)

    mad = np.median(
        np.abs(values - median)
    )

    sigma = 1.4826 * mad

    if not np.isfinite(sigma) or sigma <= 0:
        raise RuntimeError(
            f"Invalid robust sigma estimate: {sigma}"
        )

    return float(sigma)


def select_best_event(events, target_freq):
    """
    Select one find_event candidate near the injection.

    Primary criterion:
        smallest frequency separation

    Tie-breaker:
        highest reported SNR
    """
    if events.empty:
        return None, None, None, None

    frequency_column = find_column(
        events,
        [
            "Corrected_Frequency",
            "Uncorrected_Frequency",
            "Frequency",
            "Freq",
        ],
    )

    drift_column = find_column(
        events,
        [
            "Drift_Rate",
            "DriftRate",
            "drift_rate",
        ],
    )

    snr_column = find_column(
        events,
        [
            "SNR",
            "snr",
        ],
    )

    if frequency_column is None:
        print(
            "Could not identify a frequency column "
            "in the event CSV."
        )

        return None, None, None, None

    candidates = events.copy()

    candidates[frequency_column] = pd.to_numeric(
        candidates[frequency_column],
        errors="coerce",
    )

    candidates = candidates[
        np.isfinite(
            candidates[frequency_column]
        )
    ].copy()

    if candidates.empty:
        return (
            None,
            frequency_column,
            drift_column,
            snr_column,
        )

    candidates["frequency_difference_MHz"] = np.abs(
        candidates[frequency_column]
        - target_freq
    )

    candidates = candidates[
        candidates["frequency_difference_MHz"]
        <= EVENT_MATCH_TOLERANCE_MHZ
    ].copy()

    if candidates.empty:
        return (
            None,
            frequency_column,
            drift_column,
            snr_column,
        )

    if snr_column is not None:
        candidates[snr_column] = pd.to_numeric(
            candidates[snr_column],
            errors="coerce",
        )

        candidates["_sort_snr"] = (
            candidates[snr_column]
            .fillna(-np.inf)
        )

        candidates = candidates.sort_values(
            [
                "frequency_difference_MHz",
                "_sort_snr",
            ],
            ascending=[
                True,
                False,
            ],
        )

    else:
        candidates = candidates.sort_values(
            "frequency_difference_MHz"
        )

    best_event = candidates.iloc[0]

    return (
        best_event,
        frequency_column,
        drift_column,
        snr_column,
    )


def get_event_values(
    event,
    frequency_column,
    drift_column,
    snr_column,
):
    """Extract numeric frequency, drift rate, and SNR values."""
    if event is None:
        return None, None, None

    event_frequency = pd.to_numeric(
        event[frequency_column],
        errors="coerce",
    )

    if drift_column is not None:
        drift_rate = pd.to_numeric(
            event[drift_column],
            errors="coerce",
        )
    else:
        drift_rate = 0.0

    if snr_column is not None:
        event_snr = pd.to_numeric(
            event[snr_column],
            errors="coerce",
        )
    else:
        event_snr = np.nan

    if not np.isfinite(event_frequency):
        event_frequency = None

    if not np.isfinite(drift_rate):
        drift_rate = 0.0

    if not np.isfinite(event_snr):
        event_snr = None

    return (
        event_frequency,
        drift_rate,
        event_snr,
    )


# ============================================================
# Run find_event over every SNR and cadence
# ============================================================

snr_dirs = sorted(
    directory
    for directory in H5_ROOT.glob("snr_*")
    if directory.is_dir()
)

print("=" * 80)
print("RUNNING FIND_EVENT")
print("=" * 80)
print(f"Found {len(snr_dirs)} SNR directories")


for snr_dir in snr_dirs:
    snr_name = snr_dir.name

    cadence_dirs = sorted(
        directory
        for directory in snr_dir.glob("cadence_*")
        if directory.is_dir()
    )

    print()
    print("=" * 80)
    print(f"SNR DIRECTORY: {snr_name}")
    print("=" * 80)

    for cadence_dir in cadence_dirs:
        cadence_name = cadence_dir.name

        print()
        print("-" * 80)
        print(f"{snr_name} / {cadence_name}")
        print("-" * 80)

        h5_files, dat_files = get_ordered_cadence_files(
            snr_name=snr_name,
            cadence_name=cadence_name,
        )

        try:
            validate_file_pairing(
                h5_files=h5_files,
                dat_files=dat_files,
            )

        except FileNotFoundError as error:
            print(error)
            print("Skipping this cadence.")
            continue

        cadence_event_dir = (
            EVENT_ROOT
            / snr_name
            / cadence_name
        )

        cadence_event_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        dat_list = (
            cadence_event_dir
            / "dat_files.lst"
        )

        h5_list = (
            cadence_event_dir
            / "h5_files.lst"
        )

        event_csv = (
            cadence_event_dir
            / "events.csv"
        )

        write_file_list(
            dat_files,
            dat_list,
        )

        write_file_list(
            h5_files,
            h5_list,
        )

        if (
            event_csv.exists()
            and not RERUN_FIND_EVENT
        ):
            print(
                f"Existing event CSV found: {event_csv}"
            )
            continue

        try:
            events = find_event_pipeline(
                dat_file_list_str=str(dat_list),
                h5_file_list_str=str(h5_list),
                number_in_cadence=FILES_PER_CADENCE,
                on_off_first="ON",
                check_zero_drift=True,
                filter_threshold=1,
                saving=True,
                csv_name=str(event_csv),
            )

            if isinstance(events, pd.DataFrame):
                print(
                    f"Filtered events found: {len(events)}"
                )
            else:
                print("find_event completed.")

        except Exception as error:
            print(
                f"find_event failed for "
                f"{snr_name}/{cadence_name}:"
            )
            print(error)


# ============================================================
# Plot filtered Seticore events
# ============================================================

print()
print("=" * 80)
print("PLOTTING FILTERED SETICORE EVENTS")
print("=" * 80)


for injected_snr in PLOT_SNRS:
    snr_name = snr_directory_name(
        injected_snr
    )

    snr_h5_dir = (
        H5_ROOT
        / snr_name
    )

    if not snr_h5_dir.exists():
        print(
            f"Missing SNR directory: {snr_h5_dir}"
        )
        continue

    cadence_dirs = sorted(
        directory
        for directory in snr_h5_dir.glob("cadence_*")
        if directory.is_dir()
    )

    for cadence_dir in cadence_dirs:
        cadence_name = cadence_dir.name

        print()
        print("=" * 80)
        print(
            f"PLOTTING {snr_name} / {cadence_name}"
        )
        print("=" * 80)

        h5_files, dat_files = get_ordered_cadence_files(
            snr_name=snr_name,
            cadence_name=cadence_name,
        )

        try:
            validate_file_pairing(
                h5_files=h5_files,
                dat_files=dat_files,
            )

        except FileNotFoundError as error:
            print(error)
            print("Skipping plot.")
            continue

        event_csv = (
            EVENT_ROOT
            / snr_name
            / cadence_name
            / "events.csv"
        )

        events = load_events(
            event_csv
        )

        target_freq = get_injection_frequency(
            h5_files
        )

        print(
            f"Injected frequency: "
            f"{target_freq:.9f} MHz"
        )

        print(
            f"Total filtered event rows: "
            f"{len(events)}"
        )

        (
            best_event,
            frequency_column,
            drift_column,
            snr_column,
        ) = select_best_event(
            events=events,
            target_freq=target_freq,
        )

        (
            event_frequency,
            event_drift,
            event_snr,
        ) = get_event_values(
            event=best_event,
            frequency_column=frequency_column,
            drift_column=drift_column,
            snr_column=snr_column,
        )

        if best_event is None:
            print(
                "No matching find_event candidate found within "
                f"±{EVENT_MATCH_TOLERANCE_MHZ} MHz."
            )

        else:
            frequency_difference_khz = (
                abs(
                    event_frequency
                    - target_freq
                )
                * 1e3
            )

            print("Best matching event:")
            print(
                f"  Frequency : "
                f"{event_frequency:.9f} MHz"
            )
            print(
                f"  Difference: "
                f"{frequency_difference_khz:.3f} kHz"
            )
            print(
                f"  Drift     : "
                f"{event_drift:.6f} Hz/s"
            )

            if event_snr is not None:
                print(
                    f"  SNR       : "
                    f"{event_snr:.3f}"
                )

        # ----------------------------------------------------
        # Load all six waterfall crops
        # ----------------------------------------------------

        scan_data = []

        load_failed = False

        for h5_file in h5_files:
            try:
                (
                    crop,
                    crop_frequencies,
                    times,
                    source_name,
                ) = load_waterfall_crop(
                    filename=h5_file,
                    target_freq=target_freq,
                    zoom_mhz=ZOOM_MHZ,
                )

                residual = remove_broadband_background(
                    crop
                )

                scan_data.append(
                    {
                        "crop": crop,
                        "residual": residual,
                        "frequencies": crop_frequencies,
                        "times": times,
                        "source_name": source_name,
                    }
                )

            except Exception as error:
                print(
                    f"Could not load {h5_file.name}: "
                    f"{error}"
                )

                load_failed = True
                break

        if load_failed:
            print("Skipping this plot.")
            continue

        # ----------------------------------------------------
        # Estimate one common noise scale from OFF scans
        # ----------------------------------------------------

        off_residual_pixels = []

        for scan_index, scan in enumerate(scan_data):
            if scan_index % 2 == 0:
                continue

            finite_pixels = scan["residual"][
                np.isfinite(
                    scan["residual"]
                )
            ]

            off_residual_pixels.extend(
                finite_pixels
            )

        if not off_residual_pixels:
            print(
                "No valid OFF residual pixels found. "
                "Skipping this plot."
            )
            continue

        try:
            sigma_off = robust_sigma(
                off_residual_pixels
            )

        except RuntimeError as error:
            print(error)
            print("Skipping this plot.")
            continue

        print(
            f"Robust OFF-derived sigma: "
            f"{sigma_off:.3e}"
        )

        # ----------------------------------------------------
        # Convert all six scans to common sigma units
        # ----------------------------------------------------

        for scan in scan_data:
            scan["image"] = (
                scan["residual"]
                / sigma_off
            )

        # ----------------------------------------------------
        # Create figure
        # ----------------------------------------------------

        fig, axes = plt.subplots(
            FILES_PER_CADENCE,
            1,
            figsize=(
                FIGURE_WIDTH,
                FIGURE_HEIGHT,
            ),
            sharex=True,
        )

        im = None

        for scan_index, ax in enumerate(axes):
            scan_type = (
                "ON"
                if scan_index % 2 == 0
                else "OFF"
            )

            scan = scan_data[
                scan_index
            ]

            image = scan["image"]
            crop_frequencies = scan["frequencies"]
            times = scan["times"]
            source_name = scan["source_name"]

            im = ax.imshow(
                image,
                aspect="auto",
                origin="upper",
                interpolation="nearest",
                extent=[
                    crop_frequencies.max(),
                    crop_frequencies.min(),
                    times[-1],
                    times[0],
                ],
                vmin=VMIN_SIGMA,
                vmax=VMAX_SIGMA,
            )

            # ------------------------------------------------
            # Injected frequency
            # ------------------------------------------------

            ax.axvline(
                target_freq,
                linestyle="--",
                linewidth=INJECTION_LINEWIDTH,
                label="Injected frequency",
                zorder=4,
            )

            # ------------------------------------------------
            # Recovered event: draw only on ON scans
            # ------------------------------------------------

            if (
                event_frequency is not None
                and scan_index % 2 == 0
            ):
                centered_times = (
                    times
                    - times[len(times) // 2]
                )

                drift_track = (
                    event_frequency
                    + event_drift
                    * centered_times
                    / 1e6
                )

                ax.plot(
                    drift_track,
                    times,
                    linestyle="-",
                    linewidth=EVENT_LINEWIDTH,
                    label="Recovered Seticore event",
                    zorder=5,
                )

                ax.scatter(
                    event_frequency,
                    times[len(times) // 2],
                    marker="x",
                    s=EVENT_MARKER_SIZE,
                    linewidths=3.0,
                    zorder=6,
                )

            # ------------------------------------------------
            # Mark clipped panels
            # ------------------------------------------------

            panel_peak_sigma = np.nanmax(
                image
            )

            if panel_peak_sigma > VMAX_SIGMA:
                ax.text(
                    0.5,
                    0.05,
                    (
                        f"Peak pixel: "
                        f"{panel_peak_sigma:.1f}σ "
                        f"(display clipped at "
                        f"{VMAX_SIGMA:.0f}σ)"
                    ),
                    transform=ax.transAxes,
                    fontsize=12,
                    fontweight="bold",
                    horizontalalignment="center",
                    verticalalignment="bottom",
                    bbox={
                        "facecolor": "white",
                        "edgecolor": "black",
                        "alpha": 0.82,
                        "boxstyle": "round,pad=0.28",
                    },
                    zorder=10,
                )

            # ------------------------------------------------
            # ON/OFF label
            # ------------------------------------------------

            ax.text(
                0.018,
                0.82,
                f"{scan_type} {scan_index + 1}",
                transform=ax.transAxes,
                fontsize=SCAN_LABEL_SIZE,
                fontweight="bold",
                verticalalignment="top",
                bbox={
                    "facecolor": "white",
                    "edgecolor": "black",
                    "alpha": 0.88,
                    "boxstyle": "round,pad=0.35",
                },
                zorder=10,
            )

            # ------------------------------------------------
            # Source label
            # ------------------------------------------------

            ax.text(
                0.982,
                0.82,
                source_name,
                transform=ax.transAxes,
                fontsize=SOURCE_LABEL_SIZE,
                fontweight="bold",
                horizontalalignment="right",
                verticalalignment="top",
                bbox={
                    "facecolor": "white",
                    "edgecolor": "black",
                    "alpha": 0.82,
                    "boxstyle": "round,pad=0.32",
                },
                zorder=10,
            )

            ax.set_ylabel(
                "Time (s)",
                fontsize=AXIS_LABEL_SIZE,
                fontweight="bold",
                labelpad=10,
            )

            ax.tick_params(
                axis="both",
                which="major",
                labelsize=TICK_LABEL_SIZE,
                width=1.6,
                length=7,
            )

            # Only show the legend on the first panel.
            if scan_index == 0:
                handles, labels = (
                    ax.get_legend_handles_labels()
                )

                unique_handles = {}

                for handle, label in zip(
                    handles,
                    labels,
                ):
                    unique_handles[label] = handle

                ax.legend(
                    unique_handles.values(),
                    unique_handles.keys(),
                    fontsize=LEGEND_SIZE,
                    loc="upper center",
                    bbox_to_anchor=(0.5, 1.28),
                    ncol=2,
                    framealpha=0.95,
                )

        axes[-1].set_xlabel(
            "Frequency (MHz)",
            fontsize=AXIS_LABEL_SIZE,
            fontweight="bold",
            labelpad=12,
        )

        # ----------------------------------------------------
        # Colorbar
        # ----------------------------------------------------

        if im is not None:
            colorbar = fig.colorbar(
                im,
                ax=axes,
                pad=0.018,
                fraction=0.028,
            )

            colorbar.set_label(
                "Narrowband excess / OFF noise σ",
                fontsize=COLORBAR_LABEL_SIZE,
                fontweight="bold",
                labelpad=12,
            )

            colorbar.ax.tick_params(
                labelsize=TICK_LABEL_SIZE,
                width=1.4,
                length=6,
            )

        # ----------------------------------------------------
        # Event summary
        # ----------------------------------------------------

        if event_frequency is not None:
            event_summary = (
                f"Recovered event: "
                f"{event_frequency:.9f} MHz"
                f"   |   Drift: "
                f"{event_drift:.3f} Hz/s"
            )

            if event_snr is not None:
                event_summary += (
                    f"   |   Seticore SNR: "
                    f"{event_snr:.2f}"
                )

        else:
            event_summary = (
                "No find_event candidate recovered near "
                "the injected frequency"
            )

        fig.text(
            0.5,
            0.955,
            event_summary,
            ha="center",
            va="center",
            fontsize=EVENT_LABEL_SIZE,
            fontweight="bold",
            bbox={
                "facecolor": "white",
                "edgecolor": "black",
                "alpha": 0.92,
                "boxstyle": "round,pad=0.45",
            },
        )

        # ----------------------------------------------------
        # Main title
        # ----------------------------------------------------

        fig.suptitle(
            (
                "Seticore Zero-Drift Injection Recovery\n"
                f"Injected local SNR = {injected_snr}  •  "
                f"{cadence_name.replace('_', ' ').title()}"
            ),
            fontsize=TITLE_SIZE,
            fontweight="bold",
            y=0.995,
        )

        fig.subplots_adjust(
            top=0.91,
            bottom=0.06,
            left=0.085,
            right=0.90,
            hspace=0.11,
        )

        # ----------------------------------------------------
        # Save figure
        # ----------------------------------------------------

        if SAVE_FIGURES:
            output_figure_dir = (
                FIGURE_ROOT
                / snr_name
            )

            output_figure_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            output_figure = (
                output_figure_dir
                / f"{snr_name}_{cadence_name}.png"
            )

            fig.savefig(
                output_figure,
                dpi=180,
                bbox_inches="tight",
            )

            print(
                f"Saved figure: {output_figure}"
            )

        plt.show()
        plt.close(fig)

In [ ]:
%matplotlib inline

from pathlib import Path
import warnings

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Directories
# ============================================================

BASE = Path(
    "/datax/scratch/wlll2x/carmen_midres"
)

# Injected H5 files
H5_ROOT = (
    BASE
    / "seticore_snr_sweep"
)

# Seticore .dat files
SETICORE_ROOT = (
    BASE
    / "seticore_snr_cutoff_results"
)

# Analysis output
ANALYSIS_ROOT = (
    BASE
    / "seticore_snr_custom_find_event_analysis"
)

ANALYSIS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Parameters
# ============================================================

DEFAULT_TARGET_FREQ_MHZ = 1600.0

# Frequency window used to decide whether a Seticore hit
# corresponds to the injected signal.
MATCH_TOLERANCE_MHZ = 0.01

FILES_PER_CADENCE = 6

# ON / OFF / ON / OFF / ON / OFF
ON_INDICES = {0, 2, 4}
OFF_INDICES = {1, 3, 5}

SAVE_FIGURES = True
SHOW_FIGURES = True


# ============================================================
# Plot styling
# ============================================================

TITLE_SIZE = 20
AXIS_LABEL_SIZE = 16
TICK_SIZE = 13
LEGEND_SIZE = 12


# ============================================================
# Parse injected SNR
# ============================================================

def parse_injected_snr(directory_name):
    """
    Convert directory names:

        snr_020 -> 20
        snr_025 -> 25
        snr_030 -> 30
        ...
        snr_075 -> 75
    """

    value = directory_name.removeprefix(
        "snr_"
    )

    return float(value)


# ============================================================
# Read original cadence ordering
# ============================================================

def get_original_cadence_files(
    cadence_name,
):
    """
    Read the original six observations from all.lst.
    """

    lst_file = (
        BASE
        / cadence_name
        / "all.lst"
    )

    if not lst_file.exists():

        raise FileNotFoundError(
            f"Missing cadence list: "
            f"{lst_file}"
        )

    files = [
        Path(line.strip())
        for line
        in lst_file.read_text().splitlines()
        if line.strip()
    ]

    return files


# ============================================================
# Injected H5 paths
# ============================================================

def get_injected_h5_files(
    snr_name,
    cadence_name,
):
    """
    Return injected H5 files in the original cadence order.
    """

    original_files = (
        get_original_cadence_files(
            cadence_name
        )
    )

    h5_directory = (
        H5_ROOT
        / snr_name
        / cadence_name
    )

    return [
        h5_directory
        / f"{original.stem}_injected.h5"

        for original
        in original_files
    ]


# ============================================================
# Read injection frequency
# ============================================================

def get_injection_frequency(
    snr_name,
    cadence_name,
):
    """
    Read the injected frequency from an ON H5 file.

    Falls back to 1600 MHz if metadata cannot be found.
    """

    h5_files = (
        get_injected_h5_files(
            snr_name,
            cadence_name,
        )
    )

    for scan_index, filename in enumerate(
        h5_files
    ):

        if scan_index not in ON_INDICES:
            continue

        if not filename.exists():
            continue

        try:

            with h5py.File(
                filename,
                "r",
            ) as h5:

                attrs = (
                    h5["data"].attrs
                )

                if (
                    "injection_frequency_MHz"
                    in attrs
                ):

                    return float(
                        attrs[
                            "injection_frequency_MHz"
                        ]
                    )

                if (
                    "injection_frequency_Hz"
                    in attrs
                ):

                    return (
                        float(
                            attrs[
                                "injection_frequency_Hz"
                            ]
                        )
                        / 1e6
                    )

        except Exception as error:

            warnings.warn(
                f"Could not read injection metadata "
                f"from {filename}: {error}"
            )

    return DEFAULT_TARGET_FREQ_MHZ


# ============================================================
# Read one Seticore .dat file
# ============================================================

def read_seticore_dat(
    dat_file,
):
    """
    Read hit rows from a Seticore .dat file.
    """

    columns = [
        "hit_number",
        "drift_rate_Hz_s",
        "snr",
        "uncorrected_frequency_MHz",
        "corrected_frequency_MHz",
        "index",
        "freq_start_MHz",
        "freq_end_MHz",
        "sefd",
        "sefd_freq",
        "coarse_channel",
        "full_number_of_hits",
    ]

    rows = []

    if not dat_file.exists():

        return pd.DataFrame(
            columns=columns
        )

    try:

        with open(
            dat_file,
            "r",
            errors="replace",
        ) as handle:

            for line in handle:

                line = line.strip()

                if not line:
                    continue

                parts = line.split()

                # Real Seticore hit rows begin with
                # an integer hit number.
                try:

                    hit_number = int(
                        parts[0]
                    )

                except (
                    ValueError,
                    IndexError,
                ):

                    continue

                if len(parts) < 12:

                    warnings.warn(
                        f"Malformed Seticore row in "
                        f"{dat_file}:\n{line}"
                    )

                    continue

                try:

                    rows.append(
                        {
                            "hit_number":
                                hit_number,

                            "drift_rate_Hz_s":
                                float(parts[1]),

                            "snr":
                                float(parts[2]),

                            "uncorrected_frequency_MHz":
                                float(parts[3]),

                            "corrected_frequency_MHz":
                                float(parts[4]),

                            "index":
                                int(parts[5]),

                            "freq_start_MHz":
                                float(parts[6]),

                            "freq_end_MHz":
                                float(parts[7]),

                            "sefd":
                                float(parts[8]),

                            "sefd_freq":
                                float(parts[9]),

                            "coarse_channel":
                                int(parts[10]),

                            "full_number_of_hits":
                                int(parts[11]),
                        }
                    )

                except ValueError:

                    warnings.warn(
                        f"Could not parse Seticore "
                        f"row in {dat_file}:\n{line}"
                    )

    except Exception as error:

        warnings.warn(
            f"Could not read "
            f"{dat_file}: {error}"
        )

    return pd.DataFrame(
        rows,
        columns=columns,
    )


# ============================================================
# Find injected signal in one Seticore scan
# ============================================================

def find_injected_seticore_hit(
    dat_file,
    injection_frequency_MHz,
):
    """
    Look for the injected signal in one Seticore .dat file.

    A hit is considered a match if its corrected frequency
    lies within MATCH_TOLERANCE_MHZ of the known injection.

    If more than one hit matches, select:
        1. smallest frequency error
        2. highest SNR
    """

    hits = read_seticore_dat(
        dat_file
    )

    result = {
        "detected": False,
        "hit_count": len(hits),
        "recovered_frequency_MHz":
            np.nan,
        "frequency_error_kHz":
            np.nan,
        "recovered_seticore_snr":
            np.nan,
        "reported_drift_Hz_s":
            np.nan,
    }

    if hits.empty:
        return result

    hits = hits.copy()

    hits[
        "frequency_error_MHz"
    ] = np.abs(
        hits[
            "corrected_frequency_MHz"
        ]
        - injection_frequency_MHz
    )

    nearby = hits[
        hits[
            "frequency_error_MHz"
        ]
        <= MATCH_TOLERANCE_MHZ
    ].copy()

    if nearby.empty:
        return result

    nearby = nearby.sort_values(
        [
            "frequency_error_MHz",
            "snr",
        ],
        ascending=[
            True,
            False,
        ],
    )

    best = nearby.iloc[0]

    result.update(
        {
            "detected":
                True,

            "recovered_frequency_MHz":
                float(
                    best[
                        "corrected_frequency_MHz"
                    ]
                ),

            "frequency_error_kHz":
                float(
                    best[
                        "frequency_error_MHz"
                    ]
                )
                * 1e3,

            "recovered_seticore_snr":
                float(
                    best["snr"]
                ),

            "reported_drift_Hz_s":
                float(
                    best[
                        "drift_rate_Hz_s"
                    ]
                ),
        }
    )

    return result


# ============================================================
# OUR find_event
# ============================================================

def find_event(
    scan_matches,
):
    """
    Apply the cadence ON/OFF event logic ourselves.

    Expected cadence order:

        0 = ON
        1 = OFF
        2 = ON
        3 = OFF
        4 = ON
        5 = OFF

    For this injection experiment, an event survives when:

        - the injected signal is detected in all 3 ON scans
        - the injected signal is absent from all 3 OFF scans

    No drift-rate consistency cut is applied yet.
    """

    if len(scan_matches) != 6:

        return {
            "recovered": False,
            "on_detected": 0,
            "off_detected": 0,
            "reason":
                "Cadence does not contain six scans",
        }


    on_matches = [
        scan_matches[index]
        for index
        in sorted(ON_INDICES)
    ]

    off_matches = [
        scan_matches[index]
        for index
        in sorted(OFF_INDICES)
    ]


    number_on_detected = sum(
        bool(
            match["detected"]
        )
        for match
        in on_matches
    )

    number_off_detected = sum(
        bool(
            match["detected"]
        )
        for match
        in off_matches
    )


    all_on_detected = (
        number_on_detected == 3
    )

    no_off_detected = (
        number_off_detected == 0
    )


    recovered = (
        all_on_detected
        and
        no_off_detected
    )


    if recovered:

        reason = (
            "3/3 ON detections and "
            "0/3 OFF detections"
        )

    elif not all_on_detected:

        reason = (
            f"Only {number_on_detected}/3 "
            f"ON scans recovered"
        )

    else:

        reason = (
            f"Detected in "
            f"{number_off_detected}/3 "
            f"OFF scans"
        )


    return {
        "recovered":
            recovered,

        "on_detected":
            number_on_detected,

        "off_detected":
            number_off_detected,

        "reason":
            reason,
    }


# ============================================================
# Locate SNR directories
# ============================================================

snr_directories = sorted(
    SETICORE_ROOT.glob(
        "snr_*"
    ),
    key=lambda path:
        parse_injected_snr(
            path.name
        ),
)


print("=" * 80)
print("SETICORE + CUSTOM FIND_EVENT ANALYSIS")
print("=" * 80)

print(
    f"Seticore root : "
    f"{SETICORE_ROOT}"
)

print(
    f"Injected H5 root: "
    f"{H5_ROOT}"
)

print(
    f"SNR levels found: "
    f"{len(snr_directories)}"
)

print()


if not snr_directories:

    raise RuntimeError(
        f"No snr_* directories found "
        f"under {SETICORE_ROOT}"
    )


# ============================================================
# Containers
# ============================================================

scan_rows = []

cadence_rows = []


# ============================================================
# Analyze every injected SNR
# ============================================================

for snr_directory in snr_directories:

    snr_name = (
        snr_directory.name
    )

    injected_snr = (
        parse_injected_snr(
            snr_name
        )
    )

    cadence_directories = sorted(
        snr_directory.glob(
            "cadence_*"
        )
    )


    print()
    print("=" * 80)

    print(
        f"{snr_name} "
        f"(Injected local SNR = "
        f"{injected_snr:g})"
    )

    print(
        f"Cadences found: "
        f"{len(cadence_directories)}"
    )

    print("=" * 80)


    # ========================================================
    # Analyze each cadence
    # ========================================================

    for cadence_directory in cadence_directories:

        cadence_name = (
            cadence_directory.name
        )


        injection_frequency = (
            get_injection_frequency(
                snr_name,
                cadence_name,
            )
        )


        dat_files = sorted(
            cadence_directory.glob(
                "*.dat"
            )
        )


        if (
            len(dat_files)
            != FILES_PER_CADENCE
        ):

            warnings.warn(
                f"{snr_name} / "
                f"{cadence_name}: expected "
                f"{FILES_PER_CADENCE} "
                f".dat files but found "
                f"{len(dat_files)}"
            )


        # ----------------------------------------------------
        # Analyze every scan
        # ----------------------------------------------------

        scan_matches = []

        on_total_hits = 0
        off_total_hits = 0


        for (
            scan_index,
            dat_file,
        ) in enumerate(
            dat_files
        ):

            if scan_index in ON_INDICES:

                scan_type = "ON"

            elif scan_index in OFF_INDICES:

                scan_type = "OFF"

            else:

                scan_type = "UNKNOWN"


            match = (
                find_injected_seticore_hit(
                    dat_file,
                    injection_frequency,
                )
            )


            scan_matches.append(
                match
            )


            if scan_type == "ON":

                on_total_hits += (
                    match[
                        "hit_count"
                    ]
                )

            elif scan_type == "OFF":

                off_total_hits += (
                    match[
                        "hit_count"
                    ]
                )


            scan_rows.append(
                {
                    "snr_directory":
                        snr_name,

                    "injected_local_snr":
                        injected_snr,

                    "cadence":
                        cadence_name,

                    "scan_index":
                        scan_index,

                    "scan_type":
                        scan_type,

                    "filename":
                        dat_file.name,

                    "injection_frequency_MHz":
                        injection_frequency,

                    "total_seticore_hits":
                        match[
                            "hit_count"
                        ],

                    "injection_detected":
                        match[
                            "detected"
                        ],

                    "recovered_frequency_MHz":
                        match[
                            "recovered_frequency_MHz"
                        ],

                    "frequency_error_kHz":
                        match[
                            "frequency_error_kHz"
                        ],

                    "recovered_seticore_snr":
                        match[
                            "recovered_seticore_snr"
                        ],

                    "reported_drift_Hz_s":
                        match[
                            "reported_drift_Hz_s"
                        ],

                    "dat_file":
                        str(
                            dat_file
                        ),
                }
            )


        # ----------------------------------------------------
        # Run our find_event
        # ----------------------------------------------------

        event_result = (
            find_event(
                scan_matches
            )
        )


        number_on_detected = (
            event_result[
                "on_detected"
            ]
        )

        number_off_detected = (
            event_result[
                "off_detected"
            ]
        )


        # ----------------------------------------------------
        # Seticore recovery definitions
        # ----------------------------------------------------

        seticore_any_on = (
            number_on_detected
            >= 1
        )

        seticore_all_on = (
            number_on_detected
            == 3
        )


        # ----------------------------------------------------
        # Save cadence-level result
        # ----------------------------------------------------

        cadence_rows.append(
            {
                "snr_directory":
                    snr_name,

                "injected_local_snr":
                    injected_snr,

                "cadence":
                    cadence_name,

                "injection_frequency_MHz":
                    injection_frequency,

                "on_scans_detected":
                    number_on_detected,

                "off_scans_detected":
                    number_off_detected,

                "seticore_any_on":
                    seticore_any_on,

                "seticore_all_on":
                    seticore_all_on,

                "find_event_recovered":
                    event_result[
                        "recovered"
                    ],

                "find_event_reason":
                    event_result[
                        "reason"
                    ],

                "total_on_hits":
                    on_total_hits,

                "total_off_hits":
                    off_total_hits,
            }
        )


        print(
            f"{cadence_name}: "
            f"ON {number_on_detected}/3, "
            f"OFF {number_off_detected}/3, "
            f"find_event = "
            f"{event_result['recovered']}"
        )


# ============================================================
# Build DataFrames
# ============================================================

scan_results = pd.DataFrame(
    scan_rows
)

cadence_results = pd.DataFrame(
    cadence_rows
)


# ============================================================
# Single ON-scan recovery
# ============================================================

on_scan_summary = (
    scan_results[
        scan_results[
            "scan_type"
        ] == "ON"
    ]
    .groupby(
        "injected_local_snr",
        as_index=False,
    )
    .agg(
        on_scans_tested=(
            "injection_detected",
            "count",
        ),

        on_scans_recovered=(
            "injection_detected",
            "sum",
        ),
    )
)


on_scan_summary[
    "seticore_on_scan_recovery_fraction"
] = (
    on_scan_summary[
        "on_scans_recovered"
    ]
    / on_scan_summary[
        "on_scans_tested"
    ]
)


# ============================================================
# Cadence-level recovery
# ============================================================

recovery_summary = (
    cadence_results
    .groupby(
        "injected_local_snr",
        as_index=False,
    )
    .agg(
        cadences_tested=(
            "cadence",
            "count",
        ),

        seticore_any_on_recovered=(
            "seticore_any_on",
            "sum",
        ),

        seticore_all_on_recovered=(
            "seticore_all_on",
            "sum",
        ),

        find_event_recovered=(
            "find_event_recovered",
            "sum",
        ),

        total_seticore_on_hits=(
            "total_on_hits",
            "sum",
        ),

        total_seticore_off_hits=(
            "total_off_hits",
            "sum",
        ),
    )
)


# ============================================================
# Recovery fractions
# ============================================================

recovery_summary[
    "seticore_any_on_fraction"
] = (
    recovery_summary[
        "seticore_any_on_recovered"
    ]
    / recovery_summary[
        "cadences_tested"
    ]
)


recovery_summary[
    "seticore_all_on_fraction"
] = (
    recovery_summary[
        "seticore_all_on_recovered"
    ]
    / recovery_summary[
        "cadences_tested"
    ]
)


recovery_summary[
    "find_event_recovery_fraction"
] = (
    recovery_summary[
        "find_event_recovered"
    ]
    / recovery_summary[
        "cadences_tested"
    ]
)


recovery_summary = (
    recovery_summary
    .merge(
        on_scan_summary,
        on="injected_local_snr",
        how="left",
    )
)


# ============================================================
# Save tables
# ============================================================

scan_output = (
    ANALYSIS_ROOT
    / "seticore_scan_recovery.csv"
)

cadence_output = (
    ANALYSIS_ROOT
    / "custom_find_event_cadence_recovery.csv"
)

summary_output = (
    ANALYSIS_ROOT
    / "recovery_summary_by_injected_snr.csv"
)


scan_results.to_csv(
    scan_output,
    index=False,
)

cadence_results.to_csv(
    cadence_output,
    index=False,
)

recovery_summary.to_csv(
    summary_output,
    index=False,
)


# ============================================================
# Print summary
# ============================================================

print()
print("=" * 100)
print("RECOVERY SUMMARY")
print("=" * 100)


columns_to_show = [
    "injected_local_snr",
    "cadences_tested",
    "seticore_all_on_recovered",
    "seticore_all_on_fraction",
    "find_event_recovered",
    "find_event_recovery_fraction",
    "on_scans_recovered",
    "on_scans_tested",
    "seticore_on_scan_recovery_fraction",
]


print(
    recovery_summary[
        columns_to_show
    ].to_string(
        index=False
    )
)


# ============================================================
# MAIN PLOT
#
# Injected local SNR vs cadence recovery fraction
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)


ax.plot(
    recovery_summary[
        "injected_local_snr"
    ],
    recovery_summary[
        "seticore_all_on_fraction"
    ],
    marker="o",
    linewidth=2.5,
    markersize=8,
    label="Detected by Seticore (3/3 ON)",
)


ax.plot(
    recovery_summary[
        "injected_local_snr"
    ],
    recovery_summary[
        "find_event_recovery_fraction"
    ],
    marker="s",
    linewidth=2.5,
    markersize=8,
    label="Passes find_event (3 ON, 0 OFF)",
)


ax.set_ylim(
    -0.05,
    1.05,
)


ax.set_xlabel(
    "Injected local SNR",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="bold",
)


ax.set_ylabel(
    "Cadence recovery fraction",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="bold",
)


ax.set_title(
    "Injection Recovery vs Signal Strength",
    fontsize=TITLE_SIZE,
    fontweight="bold",
)


ax.tick_params(
    labelsize=TICK_SIZE,
)


ax.grid(
    alpha=0.3,
)


ax.legend(
    fontsize=LEGEND_SIZE,
)


fig.tight_layout()


if SAVE_FIGURES:

    output = (
        ANALYSIS_ROOT
        / "cadence_recovery_vs_injected_snr.png"
    )

    fig.savefig(
        output,
        dpi=180,
        bbox_inches="tight",
    )


if SHOW_FIGURES:
    plt.show()


plt.close(
    fig
)


# ============================================================
# DIAGNOSTIC PLOT
#
# Individual ON scan vs full cadence vs find_event
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)


ax.plot(
    recovery_summary[
        "injected_local_snr"
    ],
    recovery_summary[
        "seticore_on_scan_recovery_fraction"
    ],
    marker="o",
    linewidth=2,
    markersize=7,
    label="Individual ON scans",
)


ax.plot(
    recovery_summary[
        "injected_local_snr"
    ],
    recovery_summary[
        "seticore_all_on_fraction"
    ],
    marker="s",
    linewidth=2,
    markersize=7,
    label="Seticore: 3/3 ON",
)


ax.plot(
    recovery_summary[
        "injected_local_snr"
    ],
    recovery_summary[
        "find_event_recovery_fraction"
    ],
    marker="^",
    linewidth=2,
    markersize=7,
    label="find_event: 3 ON, 0 OFF",
)


ax.set_ylim(
    -0.05,
    1.05,
)


ax.set_xlabel(
    "Injected local SNR",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="bold",
)


ax.set_ylabel(
    "Recovery fraction",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="bold",
)


ax.set_title(
    "Injection Recovery at Each Pipeline Stage",
    fontsize=TITLE_SIZE,
    fontweight="bold",
)


ax.tick_params(
    labelsize=TICK_SIZE,
)


ax.grid(
    alpha=0.3,
)


ax.legend(
    fontsize=LEGEND_SIZE,
)


fig.tight_layout()


if SAVE_FIGURES:

    output = (
        ANALYSIS_ROOT
        / "pipeline_stage_recovery_vs_injected_snr.png"
    )

    fig.savefig(
        output,
        dpi=180,
        bbox_inches="tight",
    )


if SHOW_FIGURES:
    plt.show()


plt.close(
    fig
)


# ============================================================
# Find cadences rejected specifically because of OFF detection
# ============================================================

off_rejections = cadence_results[
    cadence_results[
        "seticore_all_on"
    ]
    &
    ~cadence_results[
        "find_event_recovered"
    ]
].copy()


off_rejection_output = (
    ANALYSIS_ROOT
    / "rejected_by_off_filter.csv"
)


off_rejections.to_csv(
    off_rejection_output,
    index=False,
)


print()
print("=" * 100)
print("3/3 ON BUT REJECTED BY FIND_EVENT")
print("=" * 100)


if off_rejections.empty:

    print(
        "None."
    )

else:

    print(
        off_rejections[
            [
                "injected_local_snr",
                "cadence",
                "on_scans_detected",
                "off_scans_detected",
                "find_event_reason",
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# Final information
# ============================================================

print()
print("=" * 100)
print("ANALYSIS COMPLETE")
print("=" * 100)

print(
    f"Analysis directory : "
    f"{ANALYSIS_ROOT}"
)

print(
    f"Injected SNR levels: "
    f"{recovery_summary['injected_local_snr'].nunique()}"
)

print(
    f"Cadence tests      : "
    f"{len(cadence_results)}"
)

print(
    f"Individual scans   : "
    f"{len(scan_results)}"
)

print()

print(
    f"Scan table         : "
    f"{scan_output}"
)

print(
    f"Cadence table      : "
    f"{cadence_output}"
)

print(
    f"Summary table      : "
    f"{summary_output}"
)

print(
    f"OFF rejections     : "
    f"{off_rejection_output}"
)